## Paris MoU PSC Data 

### Data Acquisition & Parsing

βήμα-βήμα η ίδια λογική  σειρά για την ανάκτηση και κατανόηση των δεδομένωντου Paris MoU (THETIS).

Η διαδικασία περιλαμβάνει:
1. Εξερεύνηση της δομής των διαθέσιμων αρχείων (discovery)
2. Εντοπισμό των κατάλληλων αρχείων επιθεωρήσεων
3. Κατέβασμα των αρχείων ZIP
4. Ανάλυση των XML δεδομένων
5. Μετατροπή τους σε επίπεδη μορφή CSV

Τα παραγόμενα δεδομένα αποθηκεύονται στον φάκελο `API-Data/`.


In [1]:
from pathlib import Path
import requests
import json
import zipfile
import xml.etree.ElementTree as ET
import csv


In [2]:
ROOT = Path.cwd().parent
RAW_DIR = ROOT / "API-Data" / "raw"
INTERIM_DIR = ROOT / "API-Data" / "interim"

RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RAW_DIR:", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)

ROOT: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final
RAW_DIR: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw
INTERIM_DIR: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\interim


### Βήμα 1: Σύνδεση με το Paris MoU API και ανάκτηση λίστας αρχείων

- connect στο Paris MoU file server API
- authorization token
- view λίστα όλων των διαθέσιμων αρχείων
- save τη λίστα αρχείων τοπικά για περαιτέρω επεξεργασία

ώστε να αναγνωρίζουμε  αρχεία πριν προχωρήσουμε σε μαζικό κατέβασμα δεδομένων.


In [3]:
import json
from pathlib import Path
import requests

In [4]:
#API_KEY = "API_KEY_Paris_MoU"
API_KEY = "API_KEY"
API_URL = "https://fileserver.parismou.org/api"

In [5]:
# Στέλνει ένα HTTP GET request στο URL
def get_json(url):
    response = requests.get(url)

    print("\nGET:", url)
    print("Status:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))
    print("Body preview:", response.text[:300])

    response.raise_for_status()

    return response.json()


In [1]:
### IP mou gia na tn valw sto ParisMou Config

import re
import requests

# Debug: pairnoume tin IP pou vlepει o ParisMoU server (apo to response body)
def print_server_seen_ip():
    url = f"{API_URL}/{API_KEY}/getauthorizationtoken"
    response = requests.get(url, timeout=60, allow_redirects=True)

    print("\nGET:", url)
    print("Status:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))

    text = response.text
    print("Body preview:", text[:500])  # emfanizei arxiko kommati gia na doume ti gyrnaei

    # Psaxnoume mesa sto body kati san:
    # [sourceip] => 1.2.3.4  ή  [sourcejp] => 1.2.3.4
    match = re.search(r"\[source(?:ip|jp)\]\s*=>\s*([0-9]+\.[0-9]+\.[0-9]+\.[0-9]+)", text)

    if match:
        ip = match.group(1)
        return ip
    else:

        print("'Source not permitted'.")
        return None

#run
server_ip = print_server_seen_ip()



NameError: name 'API_URL' is not defined

In [7]:
### token

token_url = f"{API_URL}/{API_KEY}/getauthorizationtoken"
token_resp = get_json(token_url)

if token_resp["status"]["code"] != "success":
    raise RuntimeError("Token request failed")

token = token_resp["access_token"]
print("Got token:", token[:6], "...")



GET: https://fileserver.parismou.org/api/J25hpRL7Tbk3Ei5uxU/getauthorizationtoken
Status: 200
Content-Type: application/json
Body preview: {"access_token":"SWhGdFUzK2FzclVGN3ZMTStqSlhOd3FSN2xPblNDaFpwT0xMMUVPakF1RVFjeE80ZkJXU3BtRzUxblNBN0VEVDJGMlB1TCtuTHhuWVpIZ3dEcEIxdGdxalVYS0NsMDQ3UkJzQmgvU2VuQUpVV01KazdCV0Q2Z0Y5ak5HczJHUUI=","status":{"code":"success","message":""},"meta":{"request_path":"\/api\/J25hpRL7Tbk3Ei5uxU\/getauthorizationt
Got token: SWhGdF ...


In [8]:
# filename

filelist_url = f"{API_URL}/{token}/getfilelist"
fl_resp = get_json(filelist_url)

if fl_resp["status"]["code"] != "success":
    raise RuntimeError("Filelist request failed")

files = fl_resp["files"]
files = [x for x in files if isinstance(x, str)]
print("Total filenames:", len(files))
print("Sample:", files[:20])



GET: https://fileserver.parismou.org/api/SWhGdFUzK2FzclVGN3ZMTStqSlhOd3FSN2xPblNDaFpwT0xMMUVPakF1RVFjeE80ZkJXU3BtRzUxblNBN0VEVDJGMlB1TCtuTHhuWVpIZ3dEcEIxdGdxalVYS0NsMDQ3UkJzQmgvU2VuQUpVV01KazdCV0Q2Z0Y5ak5HczJHUUI=/getfilelist
Status: 200
Content-Type: application/json
Body preview: {"status":{"code":"success","message":""},"files":["Background_Tables_01012023.zip",0,"Background_Tables__01012024.zip",0,"Background_Tables__01012025.zip",0,"Background_Tables__01012026.zip",0,"Background_Tables__01022023.zip",0,"Background_Tables__01022024.zip",0,"Background_Tables__01022025.zip",
Total filenames: 933
Sample: ['Background_Tables_01012023.zip', 'Background_Tables__01012024.zip', 'Background_Tables__01012025.zip', 'Background_Tables__01012026.zip', 'Background_Tables__01022023.zip', 'Background_Tables__01022024.zip', 'Background_Tables__01022025.zip', 'Background_Tables__01032023.zip', 'Background_Tables__01032024.zip', 'Background_Tables__01032025.zip', 'Background_Tables__01042023.zip', '

In [9]:
##### Apothikeyw edw ola ta arxeia pou vrika sto response tou http request

out_path = RAW_DIR / "filelist.json"
out_path.write_text(json.dumps(files, indent=2), encoding="utf-8")
print(f"Saved {len(files)} filenames to {out_path}")


Saved 933 filenames to C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\filelist.json


### Αναζήτηση αρχείων (patterns) μέσα στη λίστα του API

ψάχνουμε με (patterns) για να εντοπίσουμε τι είδους αρχεία υπάρχουν (π.χ. inspections, detentions, deficiencies κλπ).
να καταλάβουμε ποια αρχεία μας ενδιαφέρουν πριν κάνουμε μαζικό κατέβασμα.


In [10]:
# Load filelist.json

filelist_path = RAW_DIR / "filelist.json"
files = json.loads(filelist_path.read_text(encoding="utf-8"))

print("Loaded filenames:", len(files))
print("Example:", files[:10])


Loaded filenames: 933
Example: ['Background_Tables_01012023.zip', 'Background_Tables__01012024.zip', 'Background_Tables__01012025.zip', 'Background_Tables__01012026.zip', 'Background_Tables__01022023.zip', 'Background_Tables__01022024.zip', 'Background_Tables__01022025.zip', 'Background_Tables__01032023.zip', 'Background_Tables__01032024.zip', 'Background_Tables__01032025.zip']


In [11]:
# Search patterns
patterns = ["inspection", "deficien", "detention", "ship", "vessel", "event", "result"]

for pat in patterns:
    hits = []  

    for f in files:
        filename = f.lower()  
        if pat in filename:   
            hits.append(f)    

    # εκτύπωση αποτελεσμάτων
    print(f"\n== {pat} ({len(hits)}) ==")
    for h in hits[:50]:       
        print(h)



== inspection (11) ==
THETIS_PSC_2015_inspections.zip
THETIS_PSC_2016_inspections.zip
THETIS_PSC_2017_inspections.zip
THETIS_PSC_2018_inspections.zip
THETIS_PSC_2019_inspections.zip
THETIS_PSC_2020_inspections.zip
THETIS_PSC_2021_inspections.zip
THETIS_PSC_2022_inspections.zip
THETIS_PSC_2023_inspections.zip
THETIS_PSC_2024_inspections.zip
THETIS_PSC_2025_inspections.zip

== deficien (0) ==

== detention (0) ==

== ship (0) ==

== vessel (0) ==

== event (0) ==

== result (0) ==


### Κατέβασμα ενός αρχείου ZIP (δοκιμαστικά)

κατεβάζουμε ένα συγκεκριμένο αρχείο (π.χ. το ZIP του 2022),
ώστε να επιβεβαιώσουμε ότι:
- το API λειτουργεί
- την μορφλη των αρχείων


In [12]:
# To arxeio (dokimastika)
FILE_NAME = "THETIS_PSC_2022_inspections.zip"


In [13]:
### tokenize

# authorization token apo API
def get_token():
    url = f"{API_URL}/{API_KEY}/getauthorizationtoken"
    r = requests.get(url, timeout=60)
    r.raise_for_status()

    # Prospathoume na parei JSON
    j = r.json()

    if j["status"]["code"] != "success":
        raise RuntimeError(j["status"]["message"])

    return j["access_token"]


In [14]:
# 1) token
token = get_token()
print("Token OK:", token[:6], "...")

# 2) download URL gia to arxeio
url = f"{API_URL}/{token}/getfile/{FILE_NAME}"
out_path = RAW_DIR / FILE_NAME

print("Downloading from:", url)
print("Saving to:", out_path)

# 3) Katevasma se chunks gia na min fortwnetai olo sti mnimi
with requests.get(url, stream=True, timeout=300) as r:
    r.raise_for_status()

    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1MB chunks
            if chunk:
                f.write(chunk)

print("Downloaded:", out_path)


Token OK: SWhGdF ...
Saving to: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\THETIS_PSC_2022_inspections.zip
Downloaded: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\THETIS_PSC_2022_inspections.zip


## περιεχομένα ZIP αρχείου

- ελέγχουμε πόσα αρχεία περιέχει
- και βλέπουμε τα πρώτα ονόματα αρχείων για να καταλάβουμε τη δομή του ZIP


In [15]:
import zipfile

ZIP_PATH = RAW_DIR / "THETIS_PSC_2022_inspections.zip"
print("ZIP PATH:", ZIP_PATH)


ZIP PATH: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\THETIS_PSC_2022_inspections.zip


In [16]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

print("Files inside zip:", len(names))
for n in names[:50]:
    print(n)


Files inside zip: 1
GetPublicFile_20221231_0345.xml


### XML αρχείο

τα πρώτα bytes του αρχείο ώστε να δούμε:
  - namespaces
  - βασικά tags
  - γενική δομή του XML

In [17]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    xml_name = z.namelist()[0]

print("XML file inside zip:", xml_name)


XML file inside zip: GetPublicFile_20221231_0345.xml


In [18]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    with z.open(xml_name) as f:
        #  prota bytes
        head = f.read(20000)

print(head.decode("utf-8", errors="replace"))


<?xml version='1.0' encoding='UTF-8'?><InspectionResultsReceipt><urn:Inspection xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:InspectionID="3252775774" urn:ReportingAuthority="NL" urn:PlaceOfInspection="NLRTM" urn:DateOfFirstVisit="2022-04-01Z" urn:DateOfFinalVisit="2022-04-01Z" urn:PSCInspectionType="DETAILED_INSPECTION"><urn:ShipParticulars xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:IMO="6605503"><urn:Name xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:Value="ZEUS" urn:EffectDate="2021-04-14Z"/><urn:Flag xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:Value="PW" urn:EffectDate="2021-04-14Z"/><urn:ShipType xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:Value="399" urn:EffectDate="2006-10-12Z"/><urn:KeelDate xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:Value="1966-03-30Z" urn:EffectDat

In [19]:
##### NOTE

# Λόγω μεγέθους XML και μνήμης --> αν μετα χρειαστώ parsing --> iterparse

### Αναζήτηση σχετικών λέξεων (detention / release) στο XML
διαβάζουμε το XML αρχείο
- ψάχνουμε για λέξεις-κλειδιά σχετικές με κράτηση (detention) --> να καταλάβουμε πώς και πού αποθηκεύεται η πληροφορία της κράτησης
μέσα στο XML


In [20]:
## keywords

NEEDLES = [
    "Detention", "detention",
    "Release", "release",
    "DateOfRelease", "DateOfDetention",
    "isDetained", "Detained",
]

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    xml_name = z.namelist()[0]
    with z.open(xml_name) as f:
        data = f.read()

print("XML file:", xml_name)
print("XML size (MB):", round(len(data) / (1024 * 1024), 2))


text = data.decode("utf-8", errors="replace")

for needle in NEEDLES:
    idx = text.find(needle)

    if idx == -1:
        print("Not found")
    else:
        print("FOUND at position", idx)

        # Emfanizoume to perivallon keimenou (context)
        start = max(0, idx - 300)
        end = min(len(text), idx + 600)
        print(text[start:end])


XML file: GetPublicFile_20221231_0345.xml
XML size (MB): 126.21
FOUND at position 2970
2023-02-28Z"/></urn:StatutoryCertificates><urn:Deficiencies xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu"><urn:Deficiency xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:DefectiveItemCode="01108" urn:NatureOfDefectCode="1000" urn:isGroundDetention="false" urn:isRORelated="false" urn:isAccidentalDamage="false"/><urn:Deficiency xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:DefectiveItemCode="01113" urn:NatureOfDefectCode="1000" urn:isGroundDetention="false" urn:isRORelated="false" urn:isAccidentalDamage="false"/><urn:Deficiency xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:DefectiveItemCode="07106" urn:NatureOfDefectCode="1082" urn:isGroundDetention="false" urn:isRORelated="false" urn:isAccidentalDamage="false"/><urn:Deficiency xmlns:urn="urn:getPublicInspection
Not found
No

### Εντοπισμός αρχείων επιθεωρήσεων για έτος (2022)

χρησιμοποιούμε τη λίστα αρχείων από το API
- φιλτράρουμε τα ονόματα που αφορούν το έτος 2022 και επιβεβαιώνουμε ποιο αρχείο περιέχει τις επιθεωρήσεις PSC για το έτος αυτό
  Πρεπει να προσδιορίσουμε με βεβαιότητα το σωστό ZIP αρχείοπριν προχωρήσουμε σε parsing.


In [21]:
filelist_path = RAW_DIR / "filelist.json"
text = filelist_path.read_text(encoding="utf-8")
files = json.loads(text)

clean_files = []

for f in files:
    if isinstance(f, str):   # έλεγχος ότι το στοιχείο είναι string
        clean_files.append(f)

files = clean_files

print("Total files loaded:", len(files))


Total files loaded: 933


In [22]:
# Kratame mono etos 2022
hits = sorted([f for f in files if "THETIS_PSC_2022" in f])

print("Found", len(hits), "files containing THETIS_PSC_2022")
for h in hits:
    print(h)


Found 1 files containing THETIS_PSC_2022
THETIS_PSC_2022_inspections.zip


###  tags και attributes του XML

σκανάρουμε ένα μέρος του XML αρχείο και εντοπίζουμε:
  - ονόματα tags
  - ονόματα attributes

εστιάζουμε σε πεδία που σχετίζονται με:
  - detention
  - duration
  - release / outcome

In [23]:
import re

# Posa bytes tou XML tha skanaroume
BYTES_TO_SCAN = 2_000_000  # ~2MB


In [24]:
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    xml_name = z.namelist()[0]
    with z.open(xml_name) as f:
        chunk = f.read(BYTES_TO_SCAN).decode("utf-8", errors="replace")

print("XML file:", xml_name)
print("Scanned bytes:", len(chunk))


XML file: GetPublicFile_20221231_0345.xml
Scanned bytes: 2000000


In [25]:
# Vriskoume onomata attributes
attrs = set(re.findall(r'\burn:([A-Za-z0-9_]+)=', chunk))

# Vriskoume onomata tags 
tags = set(re.findall(r'<\/?urn:([A-Za-z0-9_]+)\b', chunk))

print("\nTag name samples:")
for t in sorted(list(tags))[:120]:
    print(t)

print("\nAttribute name samples:")
for a in sorted(list(attrs))[:200]:
    print(a)



Tag name samples:
CallSign
Charterer
Charterers
ClassCertificate
ClassCertificates
Deficiencies
Deficiency
Detention
Flag
GrossTonnage
ISMCompany
Inspection
KeelDate
Name
ShipParticulars
ShipType
StatutoryCertificate
StatutoryCertificates

Attribute name samples:
Address
CertificateCode
City
ClassStatus
Country
DateOfExpiry
DateOfFinalVisit
DateOfFirstVisit
DateOfIssue
DateOfLastSurvey
DateOfStatus
DefectiveItemCode
DurationOfDetention
EffectDate
IMO
InspectionID
IssuingAuthority
IssuingAutorityType
Name
NatureOfDefectCode
PSCInspectionType
PlaceOfInspection
PlaceOfLastSurvey
ReportingAuthority
SurveyingAuthority
SurveyingAuthorityType
Type
Value
isAccidentalDamage
isGroundDetention
isRORelated


In [26]:
# Lekseis-kleidia pou mas endiaferoun
keywords = ["det", "out", "act", "meas", "release", "close", "end", "start", "dur", "day"]

print("\nPossible outcome / detention-related attribute names:")
for a in sorted(attrs):
    low = a.lower()
    if any(k in low for k in keywords):
        print(a)

print("\nPossible outcome / detention-related tag names:")
for t in sorted(tags):
    low = t.lower()
    if any(k in low for k in keywords):
        print(t)



Possible outcome / detention-related attribute names:
DurationOfDetention
isGroundDetention

Possible outcome / detention-related tag names:
Detention


## Εντοπισμός του πεδίου DurationOfDetention στο XML

attribute `DurationOfDetention`
-την πρώτη εμφάνισή του στο XML
-το γύρω περιεχόμενο (context)

-->  η διάρκεια κράτησης αποθηκεύεται ως attribute


In [27]:
# To attribute pou mas endiaferei
NEEDLE = 'DurationOfDetention="'


with zipfile.ZipFile(ZIP_PATH, "r") as z:
    xml_name = z.namelist()[0]
    with z.open(xml_name) as f:
        text = f.read().decode("utf-8", errors="replace")

idx = text.find(NEEDLE)

if idx == -1:
    print("No DurationOfDetention found in file.")
else:
    print("FOUND at position:", idx)
    print(text[max(0, idx - 500): idx + 800])


FOUND at position: 98539
0Z"/></urn:StatutoryCertificates><urn:Deficiencies xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu"><urn:Deficiency xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:DefectiveItemCode="02113" urn:NatureOfDefectCode="1163" urn:isGroundDetention="true" urn:isRORelated="false" urn:isAccidentalDamage="false"/></urn:Deficiencies><urn:Detention xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:Type="Deficiency" urn:DurationOfDetention="8"/></urn:Inspection><urn:Inspection xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:InspectionID="3577754042" urn:ReportingAuthority="EE" urn:PlaceOfInspection="EEPAS" urn:DateOfFirstVisit="2022-02-04Z" urn:DateOfFinalVisit="2022-02-04Z" urn:PSCInspectionType="DETAILED_INSPECTION"><urn:ShipParticulars xmlns:urn="urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu" urn:IMO="9086605"><urn:Name xmlns:urn="urn

### Parsing του 2022 XML σε επίπεδο CSV

ανοίγουμε το ZIP του 2022
- κάνουμε parsing του XML με `iterparse` (ώστε να μην φορτώσουμε όλο το αρχείο στη μνήμη)

για κάθε `Inspection` εξάγουμε βασικά πεδία πλοίου/επιθεώρησης/ελλείψεων και γράφουμε ένα “flat” CSV αρχείο στο `API-Data/interim/`

αυτο για τεστ μονο στο 2022. Μετα σε όλα τα αρχεία

In [28]:
import xml.etree.ElementTree as ET
import csv

OUT_CSV = INTERIM_DIR / "inspections_2022_flat.csv"

# Namespace tou XML
NS = {"urn": "urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu"}

print("ZIP_PATH:", ZIP_PATH)
print("OUT_CSV:", OUT_CSV)


ZIP_PATH: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\THETIS_PSC_2022_inspections.zip
OUT_CSV: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\interim\inspections_2022_flat.csv


In [29]:
# pairnei ena attribute apo ena XML element.
def attr(el, name, default=None):     
    if el is None: 
        return default

    nskey = f"{{{NS['urn']}}}{name}"
    return el.attrib.get(nskey, el.attrib.get(f"urn:{name}", el.attrib.get(name, default)))


# H first_child_attr() psaxnei to proto child tag kai pairnei ena attribute apo ekei.
# px des to ShipParticulars -> Flag -> Value

def first_child_attr(parent, child_tag, attr_name, default=None):
    if parent is None:
        return default

    child = parent.find(f"urn:{child_tag}", NS)
    if child is None:
        return default

    return attr(child, attr_name, default)


In [30]:
fieldnames = [
    "InspectionID","ReportingAuthority","PlaceOfInspection",
    "DateOfFirstVisit","DateOfFinalVisit","PSCInspectionType",
    "IMO","Flag","ShipType","KeelDate","GrossTonnage",
    "ClassIssuingAuthority","ISMCompanyIMO","ISMCompanyCountry",
    "TotalDeficiencies","DefectiveItemCodes",
    "DurationOfDetention"
]

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    xml_name = z.namelist()[0]
    print("XML inside zip:", xml_name)

    with z.open(xml_name) as f, open(OUT_CSV, "w", newline="", encoding="utf-8") as out:
        writer = csv.DictWriter(out, fieldnames=fieldnames)
        writer.writeheader()

        # Streaming parse gia na min gemisei i mnimi
        context = ET.iterparse(f, events=("end",))

        count = 0
        for event, elem in context:
            if elem.tag.endswith("Inspection"):
                inspection = elem

                row = {}
                row["InspectionID"] = attr(inspection, "InspectionID")
                row["ReportingAuthority"] = attr(inspection, "ReportingAuthority")
                row["PlaceOfInspection"] = attr(inspection, "PlaceOfInspection")
                row["DateOfFirstVisit"] = attr(inspection, "DateOfFirstVisit")
                row["DateOfFinalVisit"] = attr(inspection, "DateOfFinalVisit")
                row["PSCInspectionType"] = attr(inspection, "PSCInspectionType")

                ship = inspection.find("urn:ShipParticulars", NS)
                row["IMO"] = attr(ship, "IMO")
                row["Flag"] = first_child_attr(ship, "Flag", "Value")
                row["ShipType"] = first_child_attr(ship, "ShipType", "Value")
                row["KeelDate"] = first_child_attr(ship, "KeelDate", "Value")
                row["GrossTonnage"] = first_child_attr(ship, "GrossTonnage", "Value")

                # Class certificate 
                class_cert = inspection.find("urn:ClassCertificates/urn:ClassCertificate", NS)
                row["ClassIssuingAuthority"] = attr(class_cert, "IssuingAuthority")

                # ISM company
                ism = inspection.find("urn:ISMCompany", NS)
                row["ISMCompanyIMO"] = attr(ism, "IMO")
                row["ISMCompanyCountry"] = attr(ism, "Country")

                # Deficiencies
                def_nodes = inspection.findall("urn:Deficiencies/urn:Deficiency", NS)
                codes = []
                for d in def_nodes:
                    c = attr(d, "DefectiveItemCode")
                    if c:
                        codes.append(c)

                row["TotalDeficiencies"] = len(codes)
                row["DefectiveItemCodes"] = "|".join(codes) if codes else ""

                # Detention duration (an den yparxei -> 0)
                detention = inspection.find("urn:Detention", NS)
                dur = attr(detention, "DurationOfDetention")
                row["DurationOfDetention"] = dur if dur not in (None, "") else "0"

                writer.writerow(row)

                # Free memory
                elem.clear()

                count += 1
                if count % 20000 == 0:
                    print("Parsed inspections:", count)

print("Wrote:", OUT_CSV)
print("Total inspections parsed:", count)


XML inside zip: GetPublicFile_20221231_0345.xml
Wrote: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\interim\inspections_2022_flat.csv
Total inspections parsed: 17255


In [31]:
import pandas as pd

df_2022 = pd.read_csv(OUT_CSV)
print("Rows:", len(df_2022))
print(df_2022.head())
print("\nDurationOfDetention value counts (top):")
print(df_2022["DurationOfDetention"].value_counts().head(10))

df_2022.head(10)


Rows: 17255
   InspectionID ReportingAuthority PlaceOfInspection DateOfFirstVisit  \
0    3252775774                 NL             NLRTM      2022-04-01Z   
1    3519918103                 PL             PLGDY      2022-01-05Z   
2    3529254960                 ES             ESHUV      2022-01-10Z   
3    3529574569                 IE             IEAUG      2022-01-10Z   
4    3531915142                 NL             NLRTM      2022-01-12Z   

  DateOfFinalVisit    PSCInspectionType      IMO Flag  ShipType     KeelDate  \
0      2022-04-01Z  DETAILED_INSPECTION  6605503   PW       399  1966-03-30Z   
1      2022-01-05Z  EXPANDED_INSPECTION  8416798   MD       360  1984-12-12Z   
2      2022-01-10Z   INITIAL_INSPECTION  9370305   NL       360  2006-12-22Z   
3      2022-01-10Z   INITIAL_INSPECTION  9802695   NL       360  2015-11-23Z   
4      2022-01-12Z  DETAILED_INSPECTION  9345714   RU       360  2004-12-10Z   

   GrossTonnage  ClassIssuingAuthority  ISMCompanyIMO ISMCompanyCoun

,InspectionID,ReportingAuthority,PlaceOfInspection,DateOfFirstVisit,DateOfFinalVisit,PSCInspectionType,IMO,Flag,ShipType,KeelDate,GrossTonnage,ClassIssuingAuthority,ISMCompanyIMO,ISMCompanyCountry,TotalDeficiencies,DefectiveItemCodes,DurationOfDetention
0,3252775774,NL,NLRTM,2022-04-01Z,2022-04-01Z,DETAILED_INSPECTION,6605503,PW,399,1966-03-30Z,186,151.0,NaN,NaN,14,01108|01113|07106|01113|01310|18420|02107|1011...,0
1,3519918103,PL,PLGDY,2022-01-05Z,2022-01-05Z,EXPANDED_INSPECTION,8416798,MD,360,1984-12-12Z,1543,247.0,1709202.0,NO,18,01314|01101|Other|01315|14108|07125|08108|1842...,0
2,3529254960,ES,ESHUV,2022-01-10Z,2022-01-10Z,INITIAL_INSPECTION,9370305,NL,360,2006-12-22Z,2545,115.0,5625420.0,NL,0,NaN,0
3,3529574569,IE,IEAUG,2022-01-10Z,2022-01-10Z,INITIAL_INSPECTION,9802695,NL,360,2015-11-23Z,8827,115.0,303304.0,NL,0,NaN,0
4,3531915142,NL,NLRTM,2022-01-12Z,2022-01-12Z,DETAILED_INSPECTION,9345714,RU,360,2004-12-10Z,4970,191.0,555584.0,RU,7,10105|07125|15150|07108|04103|03108|11107,0
5,3533017884,PL,PLGDY,2022-01-12Z,2022-01-12Z,DETAILED_INSPECTION,9671436,NL,360,2012-11-07Z,5460,115.0,5216692.0,NL,3,08107|14108|04103,0
6,3533058769,DK,DKSVE,2022-01-12Z,2022-01-12Z,INITIAL_INSPECTION,9321392,PT,360,2004-07-05Z,4266,186.0,1114894.0,NO,0,NaN,0
7,3533058773,DK,DKVOR,2022-01-12Z,2022-01-12Z,DETAILED_INSPECTION,9904766,NO,360,2018-12-17Z,5699,186.0,1114894.0,NO,0,NaN,0
8,3531916314,ES,ESCAS,2022-01-12Z,2022-01-12Z,INITIAL_INSPECTION,9352353,PT,360,2004-07-17Z,4990,278.0,5964139.0,NL,0,NaN,0
9,3532967926,DE,DEMUK,2022-01-12Z,2022-01-12Z,DETAILED_INSPECTION,9695963,NO,376,2014-06-23Z,12433,128.0,5389339.0,NO,0,NaN,0


### επιβεβαίωση του parsed CSV

CSV που δημιουργήθηκε από το parsing -->ιβεβαιώνουμε ότι υπάρχουν πράγματι inspections με detention
  - αριθμό γραμμών
  - διαθέσιμες στήλες
  - κατανομή της διάρκειας κράτησης


In [32]:
csv_path = INTERIM_DIR / "inspections_2022_flat.csv"
df = pd.read_csv(csv_path)

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

Rows: 17255
Columns:
['InspectionID', 'ReportingAuthority', 'PlaceOfInspection', 'DateOfFirstVisit', 'DateOfFinalVisit', 'PSCInspectionType', 'IMO', 'Flag', 'ShipType', 'KeelDate', 'GrossTonnage', 'ClassIssuingAuthority', 'ISMCompanyIMO', 'ISMCompanyCountry', 'TotalDeficiencies', 'DefectiveItemCodes', 'DurationOfDetention']


In [33]:
# times DurationOfDetention
print("\nDurationOfDetention value counts (top 15):")
print(df["DurationOfDetention"].value_counts().head(15))

# Plithos inspections me detention > 0
non_zero_det = (df["DurationOfDetention"].astype(int) > 0).sum()
print("\nNon-zero detention count:", non_zero_det)

df.head(3)


DurationOfDetention value counts (top 15):
DurationOfDetention
0     16542
3        78
4        75
5        63
7        57
6        56
8        53
2        47
10       30
15       25
9        25
11       25
12       19
13       13
17       12
Name: count, dtype: int64

Non-zero detention count: 713


,InspectionID,ReportingAuthority,PlaceOfInspection,DateOfFirstVisit,DateOfFinalVisit,PSCInspectionType,IMO,Flag,ShipType,KeelDate,GrossTonnage,ClassIssuingAuthority,ISMCompanyIMO,ISMCompanyCountry,TotalDeficiencies,DefectiveItemCodes,DurationOfDetention
0,3252775774,NL,NLRTM,2022-04-01Z,2022-04-01Z,DETAILED_INSPECTION,6605503,PW,399,1966-03-30Z,186,151.0,NaN,NaN,14,01108|01113|07106|01113|01310|18420|02107|1011...,0
1,3519918103,PL,PLGDY,2022-01-05Z,2022-01-05Z,EXPANDED_INSPECTION,8416798,MD,360,1984-12-12Z,1543,247.0,1709202.0,NO,18,01314|01101|Other|01315|14108|07125|08108|1842...,0
2,3529254960,ES,ESHUV,2022-01-10Z,2022-01-10Z,INITIAL_INSPECTION,9370305,NL,360,2006-12-22Z,2545,115.0,5625420.0,NL,0,NaN,0


In [34]:
cols = [
    "InspectionID",
    "DateOfFirstVisit",
    "PlaceOfInspection",
    "IMO",
    "Flag",
    "ShipType",
    "GrossTonnage",
    "TotalDeficiencies",
    "DurationOfDetention"
]

df.head(3)[cols]


,InspectionID,DateOfFirstVisit,PlaceOfInspection,IMO,Flag,ShipType,GrossTonnage,TotalDeficiencies,DurationOfDetention
0,3252775774,2022-04-01Z,NLRTM,6605503,PW,399,186,14,0
1,3519918103,2022-01-05Z,PLGDY,8416798,MD,360,1543,18,0
2,3529254960,2022-01-10Z,ESHUV,9370305,NL,360,2545,0,0


### Δημιουργία target μεταβλητής (duration bins)

μετατρέπουμε τη διάρκεια κράτησης (σε ημέρες) σε 9 κατηγορίες (classes)
- οι κατηγορίες είναι ίδιες με αυτές του paper
- αποθηκεύουμε dataset με τη νέα στήλη `duration_class`


In [35]:
PROCESSED_DIR = ROOT / "API-Data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

IN_CSV = INTERIM_DIR / "inspections_2022_flat.csv"
OUT_CSV = PROCESSED_DIR / "dataset_2022_with_target.csv"


In [36]:
# Metatrepei imeres kratisis (DurationOfDetention) se 9 kathgories -> Opws akribws sto paper

def bin_duration(d):
    if d <= 0:
        return "0 day"
    if d <= 2:
        return "1-2 days"
    if d == 3:
        return "3 days"
    if d == 4:
        return "4 days"
    if d == 5:
        return "5 days"
    if d <= 7:
        return "6-7 days"
    if d <= 10:
        return "8-10 days"
    if d <= 17:
        return "11-17 days"
    return "18+ days"


In [37]:
df = pd.read_csv(IN_CSV)

df["DurationOfDetention"] = pd.to_numeric(df["DurationOfDetention"], errors="coerce").fillna(0).astype(int)

df["duration_class"] = df["DurationOfDetention"].apply(bin_duration)

df.to_csv(OUT_CSV, index=False)
print("Wrote:", OUT_CSV)
print(df["duration_class"].value_counts())


Wrote: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\processed\dataset_2022_with_target.csv
duration_class
0 day         16542
6-7 days        113
11-17 days      111
18+ days        109
8-10 days       108
3 days           78
4 days           75
5 days           63
1-2 days         56
Name: count, dtype: int64


In [38]:
df.head(5)

,InspectionID,ReportingAuthority,PlaceOfInspection,DateOfFirstVisit,DateOfFinalVisit,PSCInspectionType,IMO,Flag,ShipType,KeelDate,GrossTonnage,ClassIssuingAuthority,ISMCompanyIMO,ISMCompanyCountry,TotalDeficiencies,DefectiveItemCodes,DurationOfDetention,duration_class
0,3252775774,NL,NLRTM,2022-04-01Z,2022-04-01Z,DETAILED_INSPECTION,6605503,PW,399,1966-03-30Z,186,151.0,NaN,NaN,14,01108|01113|07106|01113|01310|18420|02107|1011...,0,0 day
1,3519918103,PL,PLGDY,2022-01-05Z,2022-01-05Z,EXPANDED_INSPECTION,8416798,MD,360,1984-12-12Z,1543,247.0,1709202.0,NO,18,01314|01101|Other|01315|14108|07125|08108|1842...,0,0 day
2,3529254960,ES,ESHUV,2022-01-10Z,2022-01-10Z,INITIAL_INSPECTION,9370305,NL,360,2006-12-22Z,2545,115.0,5625420.0,NL,0,NaN,0,0 day
3,3529574569,IE,IEAUG,2022-01-10Z,2022-01-10Z,INITIAL_INSPECTION,9802695,NL,360,2015-11-23Z,8827,115.0,303304.0,NL,0,NaN,0,0 day
4,3531915142,NL,NLRTM,2022-01-12Z,2022-01-12Z,DETAILED_INSPECTION,9345714,RU,360,2004-12-10Z,4970,191.0,555584.0,RU,7,10105|07125|15150|07108|04103|03108|11107,0,0 day


### Μαζικό κατέβασμα ZIP αρχείων για όλα τα έτη (2015–2022)

authorization token από το Paris MoU API

κατεβάζουμε τα ZIP αρχεία επιθεωρήσεων για κάθε έτος 2015–2022 --> τα αρχεία στο `API-Data/raw/`

In [39]:
# apo pote mexri pote
YEARS = list(range(2015, 2023))  # 2015..2022

# filenames me naming convention
FILES = [f"THETIS_PSC_{y}_inspections.zip" for y in YEARS]

print("Total files to download:", len(FILES))
print("Example filenames:", FILES[:3], "...", FILES[-3:])


Total files to download: 8
Example filenames: ['THETIS_PSC_2015_inspections.zip', 'THETIS_PSC_2016_inspections.zip', 'THETIS_PSC_2017_inspections.zip'] ... ['THETIS_PSC_2020_inspections.zip', 'THETIS_PSC_2021_inspections.zip', 'THETIS_PSC_2022_inspections.zip']


In [40]:


# Katevazei ena ZIP arxeio kai to apothikeuei sto RAW_DIR
def download(token, filename):
    url = f"{API_URL}/{token}/getfile/{filename}"
    out_path = RAW_DIR / filename

    if out_path.exists() and out_path.stat().st_size > 0:
        print("Already exists:", filename)
        return

    print("Downloading:", filename)

    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()

        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1MB chunks
                if chunk:
                    f.write(chunk)

    print("Saved:", out_path)


In [41]:
token = get_token()
print("Token OK:", token[:6], "...")

for fn in FILES:
    download(token, fn)

print("\nDone. Files in RAW_DIR:", len(list(RAW_DIR.glob("THETIS_PSC_*_inspections.zip"))))


Token OK: SWhGdF ...
Downloading: THETIS_PSC_2015_inspections.zip
Saved: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\THETIS_PSC_2015_inspections.zip
Downloading: THETIS_PSC_2016_inspections.zip
Saved: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\THETIS_PSC_2016_inspections.zip
Downloading: THETIS_PSC_2017_inspections.zip
Saved: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\THETIS_PSC_2017_inspections.zip
Downloading: THETIS_PSC_2018_inspections.zip
Saved: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\THETIS_PSC_2018_inspections.zip
Downloading: THETIS_PSC_2019_inspections.zip
Saved: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\THETIS_PSC_2019_inspections.zip
Downloading: THETIS_PSC_2020_inspections.zip
Saved: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\raw\THETIS_PSC_2020_inspections.zip
Downloading: THETIS_PSC_2021_inspections.zip
Saved: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_F

In [42]:
# Emfanizoume megethos gia 2-3 arxeia gia epivevaiwsi
some = sorted(RAW_DIR.glob("THETIS_PSC_*_inspections.zip"))[:8]
for p in some:
    print(p.name, "MB:", round(p.stat().st_size / (1024*1024), 2))


THETIS_PSC_2015_inspections.zip MB: 5.5
THETIS_PSC_2016_inspections.zip MB: 5.57
THETIS_PSC_2017_inspections.zip MB: 5.82
THETIS_PSC_2018_inspections.zip MB: 5.79
THETIS_PSC_2019_inspections.zip MB: 5.79
THETIS_PSC_2020_inspections.zip MB: 4.29
THETIS_PSC_2021_inspections.zip MB: 5.75
THETIS_PSC_2022_inspections.zip MB: 6.48


### Parsing ΟΛΩΝ των ετών (2015–2022) σε ένα flat CSV

κάνουμε streaming parsing του XML (`iterparse`) για λόγους μνήμης
- εξάγουμε βασικά πεδία επιθεώρησης/πλοίου/ελλείψεων/κράτησης
- γράφουμε ένα ενιαίο flat dataset σε CSV --> `API-Data/interim/inspections_2015_2022_flat.csv`

Αυτό είναι το βασικό dataset που θα χρησιμοποιήσουμε στη συνέχεια για ML.


In [43]:
import xml.etree.ElementTree as ET

YEARS = list(range(2015, 2023))  # 2015..2022
ZIP_FILES = [RAW_DIR / f"THETIS_PSC_{y}_inspections.zip" for y in YEARS]

OUT_CSV = INTERIM_DIR / "inspections_2015_2022_flat.csv"

# Namespace tou XML
NS = {"urn": "urn:getPublicInspections.xmlData.business.thetis.emsa.europa.eu"}

print("parse years:", YEARS[0], "-", YEARS[-1])
print("Output CSV:", OUT_CSV)


parse years: 2015 - 2022
Output CSV: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\interim\inspections_2015_2022_flat.csv


In [44]:
# H parse_one_zip() diavazei ena ZIP arxeio, kanei streaming parse tou XML grafei mia grammi sto CSV gia kathe Inspection.

def parse_one_zip(zip_path, writer):
    if not zip_path.exists():
        print("Missing:", zip_path.name)
        return

    print("\nParsing:", zip_path.name)

    with zipfile.ZipFile(zip_path, "r") as z:
        xml_name = z.namelist()[0]

        with z.open(xml_name) as f:
            context = ET.iterparse(f, events=("end",))

            count = 0
            for event, elem in context:
                if elem.tag.endswith("Inspection"):
                    inspection = elem

                    ship = inspection.find("urn:ShipParticulars", NS)

                    # Deficiencies
                    def_nodes = inspection.findall("urn:Deficiencies/urn:Deficiency", NS)
                    codes = []
                    for d in def_nodes:
                        c = attr(d, "DefectiveItemCode")
                        if c:
                            codes.append(c)

                    # Detention duration
                    detention = inspection.find("urn:Detention", NS)
                    duration = attr(detention, "DurationOfDetention")
                    if duration is None or duration == "":
                        duration = "0"

                    # ISM Company & Class Certificate
                    ism = inspection.find("urn:ISMCompany", NS)
                    class_cert = inspection.find("urn:ClassCertificates/urn:ClassCertificate", NS)

                    row = {
                        "InspectionID": attr(inspection, "InspectionID"),
                        "ReportingAuthority": attr(inspection, "ReportingAuthority"),
                        "PlaceOfInspection": attr(inspection, "PlaceOfInspection"),
                        "DateOfFirstVisit": attr(inspection, "DateOfFirstVisit"),
                        "DateOfFinalVisit": attr(inspection, "DateOfFinalVisit"),
                        "PSCInspectionType": attr(inspection, "PSCInspectionType"),

                        "IMO": attr(ship, "IMO"),
                        "Flag": first_child_attr(ship, "Flag", "Value"),
                        "ShipType": first_child_attr(ship, "ShipType", "Value"),
                        "KeelDate": first_child_attr(ship, "KeelDate", "Value"),
                        "GrossTonnage": first_child_attr(ship, "GrossTonnage", "Value"),

                        "ClassIssuingAuthority": attr(class_cert, "IssuingAuthority"),
                        "ISMCompanyIMO": attr(ism, "IMO"),
                        "ISMCompanyCountry": attr(ism, "Country"),

                        "TotalDeficiencies": len(codes),
                        "DefectiveItemCodes": "|".join(codes) if codes else "",
                        "DurationOfDetention": duration,
                        "SourceYearZip": zip_path.name,
                    }

                    writer.writerow(row)

                    # Free memory
                    elem.clear()

                    count += 1
                    if count % 20000 == 0:
                        print("  parsed inspections:", count)

            print("  total parsed from this zip:", count)


In [45]:
## parsing for all years

fieldnames = [
    "InspectionID","ReportingAuthority","PlaceOfInspection",
    "DateOfFirstVisit","DateOfFinalVisit","PSCInspectionType",
    "IMO","Flag","ShipType","KeelDate","GrossTonnage",
    "ClassIssuingAuthority","ISMCompanyIMO","ISMCompanyCountry",
    "TotalDeficiencies","DefectiveItemCodes",
    "DurationOfDetention","SourceYearZip"
]

with open(OUT_CSV, "w", newline="", encoding="utf-8") as out:
    writer = csv.DictWriter(out, fieldnames=fieldnames)
    writer.writeheader()

    for zp in ZIP_FILES:
        parse_one_zip(zp, writer)

print("\nWrote:", OUT_CSV)



Parsing: THETIS_PSC_2015_inspections.zip
  total parsed from this zip: 17798

Parsing: THETIS_PSC_2016_inspections.zip
  total parsed from this zip: 17783

Parsing: THETIS_PSC_2017_inspections.zip
  total parsed from this zip: 17871

Parsing: THETIS_PSC_2018_inspections.zip
  total parsed from this zip: 17898

Parsing: THETIS_PSC_2019_inspections.zip
  total parsed from this zip: 17833

Parsing: THETIS_PSC_2020_inspections.zip
  total parsed from this zip: 13076

Parsing: THETIS_PSC_2021_inspections.zip
  total parsed from this zip: 15328

Parsing: THETIS_PSC_2022_inspections.zip
  total parsed from this zip: 17255

Wrote: C:\Users\georg\Desktop\Ergasia Sgouros_v0.3_Final\API-Data\interim\inspections_2015_2022_flat.csv


In [46]:
df_all = pd.read_csv(OUT_CSV)
print("Total rows:", len(df_all))
print("Years (from SourceYearZip) sample counts:")
print(df_all["SourceYearZip"].value_counts().head(10))

df_all["DurationOfDetention"] = pd.to_numeric(df_all["DurationOfDetention"], errors="coerce").fillna(0).astype(int)
print("\nNon-zero detention rows:", (df_all["DurationOfDetention"] > 0).sum())
print("\nDurationOfDetention top values:")
print(df_all["DurationOfDetention"].value_counts().head(10))

df_all.head(10)

Total rows: 134842
Years (from SourceYearZip) sample counts:
SourceYearZip
THETIS_PSC_2018_inspections.zip    17898
THETIS_PSC_2017_inspections.zip    17871
THETIS_PSC_2019_inspections.zip    17833
THETIS_PSC_2015_inspections.zip    17798
THETIS_PSC_2016_inspections.zip    17783
THETIS_PSC_2022_inspections.zip    17255
THETIS_PSC_2021_inspections.zip    15328
THETIS_PSC_2020_inspections.zip    13076
Name: count, dtype: int64

Non-zero detention rows: 4469

DurationOfDetention top values:
DurationOfDetention
0    130373
3       599
4       567
5       479
2       435
6       359
7       335
8       273
9       188
1       153
Name: count, dtype: int64


,InspectionID,ReportingAuthority,PlaceOfInspection,DateOfFirstVisit,DateOfFinalVisit,PSCInspectionType,IMO,Flag,ShipType,KeelDate,GrossTonnage,ClassIssuingAuthority,ISMCompanyIMO,ISMCompanyCountry,TotalDeficiencies,DefectiveItemCodes,DurationOfDetention,SourceYearZip
0,387577741,NL,NLRTM,2015-01-16Z,2015-01-16Z,INITIAL_INSPECTION,9374014,MT,330,2005-06-05Z,3988,115.0,4084552.0,TR,5,03104|03108|03107|11113|02108,0,THETIS_PSC_2015_inspections.zip
1,387579352,IT,ITCHI,2015-01-16Z,2015-01-22Z,DETAILED_INSPECTION,9559169,MD,360,2008-05-08Z,3068,212.0,5048711.0,UA,15,01101|11135|05107|10114|04103|10123|07115|0711...,7,THETIS_PSC_2015_inspections.zip
2,387577674,NL,NLRTM,2015-01-16Z,2015-01-16Z,DETAILED_INSPECTION,9433456,AG,353,2008-03-19Z,7464,132.0,4022178.0,DE,2,11124|13102,0,THETIS_PSC_2015_inspections.zip
3,387579865,DE,DEBRV,2015-01-16Z,2015-01-16Z,DETAILED_INSPECTION,9316024,GI,330,2005-04-12Z,2627,115.0,758614.0,GB,1,10103,0,THETIS_PSC_2015_inspections.zip
4,387578338,DE,DEBRV,2015-01-16Z,2015-01-16Z,INITIAL_INSPECTION,9377494,SE,355,2008-01-28Z,71673,160.0,5020687.0,SE,1,13102,0,THETIS_PSC_2015_inspections.zip
5,387310172,BE,BEZEE,2015-01-16Z,2015-01-16Z,INITIAL_INSPECTION,9195810,FI,360,1998-01-06Z,2876,160.0,1301668.0,FI,2,10111|02105,0,THETIS_PSC_2015_inspections.zip
6,387580946,RO,ROCND,2015-01-16Z,2015-01-16Z,EXPANDED_INSPECTION,9221891,GR,313,2000-07-24Z,59827,101.0,5137597.0,GR,2,01308|07105,0,THETIS_PSC_2015_inspections.zip
7,387599198,FR,FRNTE,2015-01-16Z,2015-01-16Z,INITIAL_INSPECTION,9405423,GR,313,2009-02-20Z,81502,160.0,1248931.0,GR,0,NaN,0,THETIS_PSC_2015_inspections.zip
8,387576720,RO,ROMID,2015-01-16Z,2015-01-16Z,DETAILED_INSPECTION,8519447,PW,360,1985-12-01Z,5548,206.0,5832761.0,RO,6,09227|07105|11131|10127|09209|09225,0,THETIS_PSC_2015_inspections.zip
9,387578286,FR,FRFOS,2015-01-16Z,2015-01-16Z,DETAILED_INSPECTION,8709767,DK,360,1987-06-15Z,852,115.0,5216499.0,DK,5,07111|10116|07123|01314|01308,0,THETIS_PSC_2015_inspections.zip
